In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving spam.csv to spam (2).csv


In [ ]:
import pandas as pd
df=pd.read_csv("spam.csv", encoding='latin1')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [ ]:
df.groupby('v1').describe()

v2                                                                 \
     count unique                                                top freq   
v1                                                                          
ham   4825   4516                             Sorry, I'll call later   30   
spam   747    653  Please call our customer service representativ...    4   

     Unnamed: 2                                                            \
          count unique                                                top   
v1                                                                          
ham          45     39   bt not his girlfrnd... G o o d n i g h t . . .@"   
spam          5      4                                        PO Box 5249   

          Unnamed: 3                                    Unnamed: 4         \
     freq      count unique                    top freq      count unique   
v1                                                                          
ham     3         10      9                     GE    2          6      5   
spam    2          2      1   MK17 92H. 450Ppw 16"    2          0      0   

                    
          top freq  
v1                  
ham   GNT:-)"    2  
spam      NaN  NaN

In [ ]:
df['v1'].value_counts()

,count
v1,
ham,4825
spam,747


In [ ]:
747/4825

0.15481865284974095

this tells there are 15% spam emails and 85%ham emails so the datset is imbalance

In [ ]:
df_spam=df[df['v1']=='spam']
df_spam.shape

(747, 5)

In [ ]:
df_ham=df[df['v1']=='ham']
df_ham.shape

(4825, 5)

In [ ]:
df_ham_downsampled=df_ham.sample(df_spam.shape[0])
df_ham_downsampled.shape

(747, 5)

In [ ]:
df_balanced=pd.concat([df_ham_downsampled,df_spam])
df_balanced.shape

(1494, 5)

In [ ]:
df_balanced['v1'].value_counts()

,count
v1,
ham,747
spam,747


In [ ]:
df_balanced['spam']=df_balanced['v1'].apply(lambda x: 1 if x=='spam' else 0)
df_balanced.sample(5)

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4,spam
3761,spam,FREE for 1st week! No1 Nokia tone 4 ur mob eve...,NaN,NaN,NaN,1
2611,spam,Knock Knock Txt whose there to 80082 to enter ...,NaN,NaN,NaN,1
3441,spam,Save money on wedding lingerie at www.bridal.p...,NaN,NaN,NaN,1
3629,spam,Get the official ENGLAND poly ringtone or colo...,NaN,NaN,NaN,1
2771,ham,Then ur sis how?,NaN,NaN,NaN,0


In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df_balanced['v2'],df_balanced['spam'],stratify=df_balanced['spam'])
X_train.head(4)

,v2
3927,Babe ? I lost you ... Will you try rebooting ?
5076,"Guy, no flash me now. If you go call me, call ..."
4865,"Oh! Shit, I thought that was your trip! Looooo..."
2429,Guess who am I?This is the first time I create...


In [ ]:
bert_preprocess = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
bert_encoder = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4")


In [ ]:
def get_sentences_embedding(sentences):
  preprocessing_text=bert_preprocess(sentences)
  return bert_encoder(preprocessed_text)['pooled_output']

In [ ]:
def get_sentences_embedding(sentences):
  preprocessing_text=bert_preprocess(sentences)
  return bert_encoder(preprocessing_text)['pooled_output']

get_sentences_embedding([
    "500$ discount,hurry up",
    "Devi i want to talk with you call me at your free time"
])

<tf.Tensor: shape=(2, 768), dtype=float32, numpy=
array([[-0.7734204 , -0.50137717, -0.8073748 , ..., -0.60576576,
        -0.7409629 ,  0.8849831 ],
       [-0.75410897, -0.2988708 ,  0.2689049 , ...,  0.10059899,
        -0.61297774,  0.79255474]], dtype=float32)>

In [ ]:
e=get_sentences_embedding([
    "banana",
    "grapes",
    "mango",
    "jeff bezos",
    "elon musk",
    "bill gates"
])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_similarity([e[0]],[e[1]])

array([[0.9911088]], dtype=float32)

In [ ]:
cosine_similarity([e[0]],[e[4]])

array([[0.893363]], dtype=float32)

There are two types of models that you can build in tensorflow
**1.Sequential,2.functional**

In [ ]:
!pip install tensorflow-text

In [ ]:
# Bert layers
text_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name='text')
preprocessed_text = bert_preprocess(text_input)
outputs = bert_encoder(preprocessed_text)

# Neural network layers
l = tf.keras.layers.Dropout(0.1, name="dropout")(outputs['pooled_output'])
l = tf.keras.layers.Dense(1, activation='sigmoid', name="output")(l)

# Use inputs and outputs to construct a final model
model = tf.keras.Model(inputs=[text_input], outputs = [l])


ValueError: Exception encountered when calling layer 'keras_layer_22' (type KerasLayer).

A KerasTensor is symbolic: it's a placeholder for a shape an a dtype. It doesn't have any actual numerical value. You cannot convert it to a NumPy array.

Call arguments received by layer 'keras_layer_22' (type KerasLayer):
  • inputs=<KerasTensor shape=(None,), dtype=string, sparse=False, ragged=False, name=text>
  • training=None

In [ ]:
import tensorflow as tf
import tensorflow_text as text

print(f"TensorFlow version: {tf.__version__}")
print(f"TensorFlow Text version: {text.__version__}")

TensorFlow version: 2.19.0
TensorFlow Text version: 2.19.0


In [ ]:
len(X_train)

In [ ]:
METRICS = [
      tf.keras.metrics.BinaryAccuracy(name='accuracy'),
      tf.keras.metrics.Precision(name='precision'),
      tf.keras.metrics.Recall(name='recall')
]


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=METRICS
)

In [ ]:
model.fit(X_train,y_train,epochs=10)

In [ ]:
y_predicted=model.predict(X_test)
y_predicted=y_predicted.flatten()

In [ ]:
from sklearn.metrics import confusion_matrix,classification_report
cm=confusion_matrix(y_test,y_predicted.round())
cm

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
sns.heatmap(cm,annot=True,fmt='d')
plt.xlabel('Predicted')
plt.ylabel('Truth')

In [ ]:
print(classification_report(y_test, y_predicted))

In [ ]:
reviews = [
    'Enter a chance to win $5000, hurry up, offer valid until march 31, 2021',
    'You are awarded a SiPix Digital Camera! call 09061221061 from landline. Delivery within 28days. T Cs Box177. M221BP. 2yr warranty. 150ppm. 16 . p pÂ£3.99',
    'it to 80488. Your 500 free text messages are valid until 31 December 2005.',
    'Hey Sam, Are you coming for a cricket game tomorrow',
    "Why don't you wait 'til at least wednesday to see if you get your ."
]
model.predict(reviews)
